In [2]:
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib
import time  # For measuring computational time

# Load the dataset
data = pd.read_csv("Synthetic Frailty Data.csv")  # Replace with the actual file path

# Select relevant columns for Grip Strength Test
data = data[["Age", "Gender", "Grip Strength (kg)", "Grip Strength Result"]]

# Encode Gender as numeric (Male = 1, Female = 0)
data["Gender"] = data["Gender"].map({"Male": 1, "Female": 0})

# Encode target variable (Grip Strength Result)
label_encoder = LabelEncoder()
data["Grip Strength Result"] = label_encoder.fit_transform(data["Grip Strength Result"])

# Define features (X) and target (y)
X = data[["Age", "Gender", "Grip Strength (kg)"]]
y = data["Grip Strength Result"]

# Standardize features for models like SVM and KNN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Save the scaler and label encoder for Grip Strength Test
joblib.dump(scaler, 'Scaler_GS.pkl')
print("Scaler saved as 'Scaler_GS.pkl'")
joblib.dump(label_encoder, 'Label_Encoder_GS.pkl')  # Save the LabelEncoder
print("Label Encoder saved as 'Label_Encoder_GS.pkl'")

# Initialize models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Support Vector Machine": SVC(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
}

# Train and evaluate models
results = {}
computation_costs = {}
saved_models = {}

for model_name, model in models.items():
    start_time = time.time()  # Start timing
    model.fit(X_train, y_train)
    end_time = time.time()  # End timing

    # Make predictions
    y_pred = model.predict(X_test)

    # Evaluate the model
    acc = accuracy_score(y_test, y_pred)
    training_time = end_time - start_time  # Training time in seconds

    # Print results for the current model
    print(f"Model: {model_name}")
    print(f"Accuracy: {acc:.2f}")
    print(f"Training Time (seconds): {training_time:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
    print("-" * 50)

    # Store results
    results[model_name] = acc
    computation_costs[model_name] = training_time

    # Save the trained model with '_GS' suffix
    model_filename = f"{model_name.replace(' ', '_')}_GS.pkl"
    joblib.dump(model, model_filename)
    saved_models[model_name] = model_filename
    print(f"Model '{model_name}' saved as '{model_filename}'")

# Compare accuracies and computational costs
print("\nModel Performance Summary:")
for model_name, acc in results.items():
    print(f"{model_name}: Accuracy = {acc:.2f}, Training Time = {computation_costs[model_name]:.4f} seconds")
    print(f"Model saved as: {saved_models[model_name]}")


Scaler saved as 'Scaler_GS.pkl'
Label Encoder saved as 'Label_Encoder_GS.pkl'
Model: Logistic Regression
Accuracy: 0.83
Training Time (seconds): 0.0195

Classification Report:
              precision    recall  f1-score   support

   Good Grip       0.75      0.75      0.75        68
    Low Grip       0.87      0.87      0.87       132

    accuracy                           0.83       200
   macro avg       0.81      0.81      0.81       200
weighted avg       0.83      0.83      0.83       200

--------------------------------------------------
Model 'Logistic Regression' saved as 'Logistic_Regression_GS.pkl'
Model: Random Forest
Accuracy: 1.00
Training Time (seconds): 0.1815

Classification Report:
              precision    recall  f1-score   support

   Good Grip       1.00      1.00      1.00        68
    Low Grip       1.00      1.00      1.00       132

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg    

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:31:49] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [ ]:
import zipfile

# Create a zip archive containing the models, scaler, and label encoder
files_to_download = [
    "Logistic_Regression_GS.pkl", "Random_Forest_GS.pkl", "Support_Vector_Machine_GS.pkl",
    "K-Nearest_Neighbors_GS.pkl", "XGBoost_GS.pkl", "Scaler_GS.pkl", "Label_Encoder_GS.pkl"
]

with zipfile.ZipFile('GS_Models.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in files_to_download:
        zipf.write(file)

print("Download the 'GS_Models.zip' file from your directory.")


Download the 'GS_Models.zip' file from your directory.
